In [1]:
import re
from itertools import zip_longest
import gc

import xml.etree.ElementTree as ET
import numpy as np
from datetime import datetime

"""
ToyNetwork 재분석
    화성~서울 네트워크(서울방향만)
    집계시간 : 10~30초
    분석시간 1800~13200 중 1800~12600 사용
    램프 ALL
    연결로 영향권3 - 150m 상류 검지기
    모형식 EL 방법론 수정
"""
import pandas as pd

import os

"""
pd.set_option("display.max_rows", None)      # 모든 행 표시
pd.set_option("display.max_columns", None)   # 모든 열 표시
pd.set_option("display.width", None)         # 출력 폭 제한 해제
pd.set_option("display.max_colwidth", None)  # 셀 내용 생략 방지
"""

# FIX 값 모음
###################################################################################################################

path = r"C:\Vissim_Workspace\시나리오(140-유고지점 Uniform Random)_260518\연결로 영향권_3차로폐쇄\영향권3_진출"

senario_path = r"C:\Users\alswl\Desktop\유고 시나리오\추정용 시나리오\synthetic_scenarios_all_140_with_set_type(영향권3(진출))_3차로폐쇄.csv"

inpx_path = r"C:\Vissim_Workspace\시나리오(140-유고지점 Uniform Random)_260518\연결로 영향권_3차로폐쇄\영향권3_진출\화성~서울(140-유고지점)_260526.inpx"

# 한번에 처리할 .mer파일 갯수
num_mer = 3

# 임계 연속 시간
continuous_list = [180]


# 집계시간(분)
#min_time = 10 #5,10,15

# 집계시간(초)
sec_time_list = [10, 20, 30]

# 상류 검지기 위치(150m, 250m, 350m)
up_dcp_pos_list = [1, 2, 3]

#대/시 환산
revision_time = None

start_interval = 1800
end_use_interval = 12600
end_interval = 13200

weights = {
    "w1" : 1,
    "w2" : 1,
    "w3" : 1,
    "w4" : 1,
    "w5" : 1,
    "w6" : 1
}

vehicle_types = [100, 300, 630, 640, 650]
###################################################################################################################
"""
    모델 모형식 변수
"""

# 유고 발생 전 차로 수
lane = None

# 유고 발생 차로 수
acc_lane = None

# 유고 발생 시간(초)
acc_start_time = 3600

# 유고 해제 시간(초)
#acc_finish_list = [3300, 3900, 4800, 5700, 6600] # 5분, 15분, 30분, 45분, 60분
acc_finish_time = None
# 종단 경사
lane_gradient = None

# 유고 지점
incident_measurement = None

# 임계시간
Vc = 53.7

# 램프 간섭 영향률
###################################################################################################################
# 램프 전 본선 검지기(램프 간섭 영향률)
before_ramp = [70, 117, 124, 186, 215]

# 램프 후 본선 검지기(램프 간섭 영향률)
after_ramp = [74, 121, 127, 190, 221]

# 유입램프 검지기(램프 간섭 영향률)
input_ramp = [902, 904]

# 유출램프 검지기(램프 간섭 영향률)
output_ramp = [901, 903, 905]

# 진출 정상성(진입)(진출 원활률)
enter_line = [23, 121, 190]

# 유입램프 바로 뒤 본선 검지기(진출 원활률)(램프 간섭 영향률)
input_main_ramp = [121, 190]

# 유출램프 바로 앞 본선 검지기(진출 원활률)(램프 간섭 영향률)
output_main_ramp = [73, 126, 220]

# 3차로 검지기
three_lane = [71, 72, 73, 119, 120, 125, 126, 188, 189, 216, 217, 218, 219, 220]
###################################################################################################################


# 함수 모음
###################################################################################################################

# 시간순으로 데이터 정렬
def sort_ascending(df):
    # 먼저 시간 순서용 StartTime 생성
    df["StartTime"] = (
        df["TimeGroup"]
        .astype(str)
        .str.split("~")
        .str[0]
        .astype(int)
    )
    df["EndTime"] = (
        df["TimeGroup"]
        .astype(str)
        .str.split("~")
        .str[1]
        .astype(int)
    )
    # 반드시 V_next 계산 전에 정렬
    df = df.sort_values(["StartTime", "EndTime", "New_Measurement"]).reset_index(drop=True)

    return df

# 속도 변화율
def speed_mean(original_df):
    copy_df = original_df.copy()

    # 램프 검지기 제외
    copy_df = copy_df[~copy_df["New_Measurement"].between(900, 910)]

    measurements = sorted(copy_df["New_Measurement"].dropna().unique())

    # TimeGroup, New_Measurement별 그룹화 및 속도 평균
    speed_mean_df = (
        copy_df.groupby(["TimeGroup", "New_Measurement"])
          .agg(V_mean=("v[km/h]", "mean"), V_count=("v[km/h]", "count"))
          .reset_index()
    )

    speed_mean_df = complete_time_measurement(
        speed_mean_df,
        value_cols=["V_mean", "V_count"],
        measurements=measurements
    )

    speed_mean_df["V_next"] = speed_mean_df.groupby("TimeGroup")["V_mean"].shift(-1)
    cols=["V_mean", "V_next"]
    speed_mean_df[cols] = speed_mean_df[cols].fillna(0)
    speed_mean_df["delta_V"] = np.where(speed_mean_df["V_mean"] == 0,
        0,
        (speed_mean_df["V_next"] - speed_mean_df["V_mean"]) / speed_mean_df["V_mean"]
    )

    return speed_mean_df

# 밀도 변화율
def density_mean(speed_df):
    density_mean_df  = speed_df.copy()

    density_mean_df["K"] = np.where(
        density_mean_df["V_mean"] == 0,
        0,
        density_mean_df["V_count"] * revision_time / density_mean_df["V_mean"]
    )
    density_mean_df["K_next"] = density_mean_df.groupby("TimeGroup")["K"].shift(-1)
    cols=["K", "K_next"]
    density_mean_df[cols] = density_mean_df[cols].fillna(0)
    density_mean_df["delta_K"] = np.where(density_mean_df["K"] == 0,
        0,
        (density_mean_df["K_next"] - density_mean_df["K"]) / density_mean_df["K"]
    )

    density_mean_df = complete_time_measurement(
        density_mean_df,
        value_cols=["V_mean", "V_count", "K", "K_next", "delta_K"]
    )
    return density_mean_df

# 중차량 혼입률
def heavy_rate(original_df):
    copy_df = original_df.copy()

    measurements = sorted(copy_df["New_Measurement"].dropna().unique())

    heavy_df = (
        copy_df.groupby(["TimeGroup", "New_Measurement"])
        .agg(
            heavy_count=("Vehicle type", lambda x: x.isin([630, 640, 650]).sum()),
            total_count=("Vehicle type", "count")
        )
        .reset_index()
    )
    heavy_df = sort_ascending(heavy_df)

    heavy_df = complete_time_measurement(
        heavy_df,
        value_cols=["heavy_count", "total_count"],
        measurements=measurements
    )

    heavy_df["rate"] = np.where(
        heavy_df["total_count"] == 0,
        0,
        heavy_df["heavy_count"] / heavy_df["total_count"]
    )

    return heavy_df

# 동적 포화도
def entry_saturation(original_df):
    copy_df = original_df.copy()

    copy_df = copy_df[~copy_df["New_Measurement"].between(900, 910)]

    measurements = sorted(copy_df["New_Measurement"].dropna().unique())

    # 실측용량 C(2차로 4400)
    entry_saturation_df = (
        copy_df.groupby(["TimeGroup", "New_Measurement"])
        .size()
        .reset_index(name="entry_volume")  # 차량 수를 entry_volume이라는 컬럼명으로
    )

    entry_saturation_df = complete_time_measurement(
        entry_saturation_df,
        value_cols=["entry_volume"],
        measurements=measurements
    )

    entry_saturation_df = sort_ascending(entry_saturation_df)

    # 행별 capacity 설정
    entry_saturation_df["capacity"] = np.where(
        entry_saturation_df["New_Measurement"].isin(three_lane),
        6600,   # 3차로
        4400    # 2차로
    )

    entry_saturation_df["Phi_진입"] = entry_saturation_df["entry_volume"] * revision_time / entry_saturation_df["capacity"]

    return entry_saturation_df

# 램프 간섭 영향률
def rfr_rate(original_df):
    copy_df = original_df.copy()
    copy_df = copy_df[copy_df["TimeGroup"].notna()]
    copy_df["TimeGroup"] = copy_df["TimeGroup"].astype(str)
    main_results=[]

    for i, (before, after) in enumerate(zip(before_ramp, after_ramp)):
        q_before = (copy_df[copy_df["New_Measurement"] == before] # 70, 117, 124, 186, 215, 312, 342, 403, 412, 460
                    .groupby("TimeGroup")
                    .size()
                    .reset_index(name="q_before"))

        q_after = (copy_df[copy_df["New_Measurement"] == after] # 74, 121, 127, 190, 221, 317, 345, 406, 416, 465
                   .groupby("TimeGroup")
                   .size()
                   .reset_index(name="q_after"))

        merged = q_before.merge(q_after, on="TimeGroup", how="outer").fillna(0)
        merged["before_ramp"] =  before
        merged["after_ramp"] =  after
        merged["Qm"] = (merged["q_before"] + merged["q_after"]) / 2
        main_results.append(merged)


    ramp_results = []
    for input_, output_ in zip_longest(input_ramp, output_ramp):

        if output_ is not None:
            q_out = (copy_df[copy_df["New_Measurement"] == output_] # 901, 903, 905
                     .groupby("TimeGroup").size().reset_index(name="q_out"))
            q_out["New_Measurement"] = output_
            ramp_results.append(q_out)

        if input_ is not None:
            q_in = (copy_df[copy_df["New_Measurement"] == input_] # 902, 904
                    .groupby("TimeGroup").size().reset_index(name="q_in"))
            q_in["New_Measurement"] = input_
            ramp_results.append(q_in)


    input_queue = input_main_ramp.copy() # 100
    output_queue = output_main_ramp.copy() #
    rfr_list = []

    for i in range(min(len(main_results), len(ramp_results))):
        main_df = main_results[i]
        ramp_df = ramp_results[i]

        rfr_df = pd.merge(main_df, ramp_df, on="TimeGroup", how="outer").fillna(0)

        # 기본값 초기화
        rfr_df["IR_in"] = 0
        rfr_df["IR_out"] = 0

        # q_in 있을 때 (유입)
        if "q_in" in rfr_df.columns:
            rfr_df["IR_in"] = np.where(
                rfr_df["Qm"] == 0,
                0,
                rfr_df["q_in"] / rfr_df["Qm"]
            )
            if input_queue:  # 남은 게 있으면 하나 꺼냄
                current_input = input_queue.pop(0)
                rfr_df["New_Measurement"] = current_input

        # q_out 있을 때 (유출)
        if "q_out" in rfr_df.columns:
            rfr_df["IR_out"] = np.where(
                rfr_df["Qm"] == 0,
                0,
                rfr_df["q_out"] / rfr_df["Qm"]
            )
            if output_queue:
                current_output = output_queue.pop(0)
                rfr_df["New_Measurement"] = current_output

        rfr_df["RFR"] = rfr_df["IR_in"] + rfr_df["IR_out"]

        rfr_list.append(rfr_df)

    if not rfr_list:
        base = copy_df[["TimeGroup"]].drop_duplicates().copy()
        all_measurements = copy_df["New_Measurement"].unique()

        expanded = []
        for m in all_measurements:
            temp = base.copy()
            temp["New_Measurement"] = m
            temp["RFR"] = 0
            expanded.append(temp)

        final_rfr_df = pd.concat(expanded, ignore_index=True)
        final_rfr_df = final_rfr_df.sort_values(by=["TimeGroup", "New_Measurement"]).reset_index(drop=True)
        final_rfr_df["RFR"] = final_rfr_df["RFR"].fillna(0)
    else :
        # -----------------------------
        # 특정 검지기에만 RFR 반영
        # -----------------------------
        final_rfr_df = pd.concat(rfr_list, ignore_index=True)

        target_measurements = input_main_ramp + output_main_ramp
        all_measurements = copy_df["New_Measurement"].unique()

        expanded_df_list = []

        base_rfr_df = final_rfr_df.copy()

        for m in all_measurements:
            if m in target_measurements:
                temp = base_rfr_df[base_rfr_df["New_Measurement"] == m].copy()
            else:
                temp = base_rfr_df[["TimeGroup"]].drop_duplicates().copy()
                temp["New_Measurement"] = m
                temp["RFR"] = 0

            expanded_df_list.append(temp)

        final_rfr_df = pd.concat(expanded_df_list, ignore_index=True)
        final_rfr_df = final_rfr_df.sort_values(by=["TimeGroup", "New_Measurement"]).reset_index(drop=True)
        final_rfr_df = final_rfr_df[["TimeGroup", "New_Measurement", "RFR"]]
        final_rfr_df["RFR"] = final_rfr_df["RFR"].fillna(0)

    final_rfr_df = sort_ascending(final_rfr_df)

    final_rfr_df = complete_time_measurement(
        final_rfr_df,
        value_cols=["RFR"]
    )
    return final_rfr_df


# 진출 원활율- output_main_ramp
def output_normality(original_df):
    copy_df = original_df.copy()

    copy_df = copy_df[copy_df["TimeGroup"].notna()]

    normality_list = []

    # 여러 진입/진출 쌍 처리
    for enter, exit_ramp, exit_main in zip(enter_line, output_ramp, output_main_ramp):
        entry_df = copy_df[copy_df["New_Measurement"] == enter][["New_Measurement", "VehNo", "t(Entry)"]]

        exit_df  = copy_df[copy_df["New_Measurement"] == exit_ramp][["New_Measurement", "VehNo", "t(Entry)"]]

        # 차량 번호별 최소 통과시각
        entry_first = (
            entry_df.groupby("VehNo")["t(Entry)"].min()
            .reset_index()
            .rename(columns={"t(Entry)": "t_entry"})
        )

        exit_first = (
            exit_df.groupby("VehNo")["t(Entry)"].min()
            .reset_index()
            .rename(columns={"t(Entry)": "t_exit"})
        )

        # 진입-진출 매칭 후 지연시간 계산
        merged = pd.merge(entry_first, exit_first, on="VehNo", how="inner")
        merged["delay_sec"] = merged["t_exit"] - merged["t_entry"]
        merged = merged[merged["delay_sec"] >= 0]  # 음수 제거

        # 중간값 기반 시간지연 bin 계산
        if len(merged) and np.isfinite(np.nanmedian(merged["delay_sec"])):
            lag_bins = int(round(np.nanmedian(merged["delay_sec"]) / sec_time))
        else:
            lag_bins = 0

        # 진입/진출 카운트 집계
        entry_count = (
            copy_df[copy_df["New_Measurement"] == enter]
            .groupby("TimeGroup")
            .size()
            .reset_index(name="Q_in")
        )

        exit_count = (
            copy_df[copy_df["New_Measurement"] == exit_ramp]
            .groupby("TimeGroup")
            .size()
            .reset_index(name="Q_out")
        )

        # 병합 후 지연만큼 shift
        merged_counts = pd.merge(entry_count, exit_count, on="TimeGroup", how="left")


        merged_counts["Q_out_shift"] = merged_counts["Q_out"].shift(-lag_bins)



        merged_counts["F(outrate)"] = np.where(
            merged_counts["Q_in"] == 0,
            0,
            merged_counts["Q_out_shift"] / merged_counts["Q_in"]
        )

        merged_counts["New_Measurement"] = exit_main  # 진출 지점에 부여

        normality_list.append(merged_counts)

    if not normality_list:
        base = copy_df[["TimeGroup"]].drop_duplicates().copy()
        all_measurements = copy_df["New_Measurement"].unique()
        expanded = []
        for m in all_measurements:
            temp = base.copy()
            temp["New_Measurement"] = m
            temp["F(outrate)"] = 0
            expanded.append(temp)

        final_df = pd.concat(expanded, ignore_index=True)
        final_df = final_df.sort_values(by=["TimeGroup", "New_Measurement"]).reset_index(drop=True)
        final_df["F(outrate)"] = final_df["F(outrate)"].fillna(0)

    else :
        # 모든 진출 램프 결과 병합
        final_df = pd.concat(normality_list, ignore_index=True)

        # 전체 검지기 확장
        all_measurements = copy_df["New_Measurement"].unique()
        expanded_list = []

        for m in all_measurements:
            if m in output_main_ramp:
                temp = final_df[final_df["New_Measurement"] == m].copy()
            else:
                temp = final_df[["TimeGroup"]].drop_duplicates().copy()
                temp["New_Measurement"] = m
                temp["F(outrate)"] = 0
            expanded_list.append(temp)

        final_df = pd.concat(expanded_list, ignore_index=True)
        final_df = final_df.sort_values(by=["TimeGroup", "New_Measurement"]).reset_index(drop=True)
    final_df = sort_ascending(final_df)

    final_df = complete_time_measurement(
        final_df,
        value_cols=["F(outrate)"]
    )
    return final_df


def calculate_stvm(speed_df, density_df, heavy_df, entry_saturation_df, rfr_df, normality_df):

    merged_df = (
        speed_df[["TimeGroup", "New_Measurement", "delta_V"]]
        .merge(
            density_df[["TimeGroup", "New_Measurement", "delta_K"]],
            on=["TimeGroup", "New_Measurement"],
            how="outer"
        )
        .merge(
            heavy_df[["TimeGroup", "New_Measurement", "rate"]],
            on=["TimeGroup", "New_Measurement"],
            how="outer"
        )
        .merge(
            entry_saturation_df[["TimeGroup", "New_Measurement", "Phi_진입"]],
            on=["TimeGroup", "New_Measurement"],
            how="outer"
        )
        .merge(
            rfr_df[["TimeGroup", "New_Measurement", "RFR"]],
            on=["TimeGroup", "New_Measurement"],
            how="outer"
        )
        .merge(
            normality_df[["TimeGroup", "New_Measurement", "F(outrate)"]],
            on=["TimeGroup", "New_Measurement"],
            how="outer"
        )
    )

    merged_df.fillna(0, inplace=True)

    merged_df["STVM"] = (
        weights["w1"] * merged_df["delta_V"] +
        weights["w2"] * merged_df["delta_K"] +
        weights["w3"] * merged_df["rate"] +
        weights["w4"] * merged_df["Phi_진입"] +
        weights["w5"] * merged_df["RFR"] +
        weights["w6"] * merged_df["F(outrate)"]
    )
    merged_df.replace([np.inf, -np.inf], 0, inplace=True)
    merged_df = modify_frame(merged_df)
    return merged_df

def calc_z(df):
    copy_df = df.copy()
    if copy_df.empty:
        return copy_df

    # 검지기별 평균
    stvm_avg_df = (
        copy_df
        .groupby("New_Measurement")["STVM"]
        .mean()
        .reset_index(name="STVM_MEAN")
    )



    avg_stvm = stvm_avg_df["STVM_MEAN"].mean()
    std_stvm = stvm_avg_df["STVM_MEAN"].std(ddof=0)

    stvm_avg_df["Z-변환"] = (stvm_avg_df["STVM_MEAN"] - avg_stvm) / std_stvm

    z_max = stvm_avg_df["Z-변환"].max()
    z_min = stvm_avg_df["Z-변환"].min()
    stvm_avg_df["환산점수"] = stvm_avg_df["Z-변환"].apply(lambda z: z_to_score(z, z_min, z_max))

    return stvm_avg_df

def calculate_z_score(stvm_df):
    copy_df = stvm_df.copy()

    # 구간 나누기
    group1 = copy_df[copy_df["New_Measurement"].between(1, 265)].copy()

    group1 = calc_z(group1)

    merged = group1.sort_values(by="New_Measurement")
    #save_to_excel(merged, folder_path, "환산점수 원시데이터", i)

    #stvm_df = pd.pivot(merged, index="TimeGroup", columns= "New_Measurement", values="환산점수")

    return merged

def modify_frame(df):
    modify_df = df.copy()
    modify_df["StartTime"] = modify_df["TimeGroup"].str.split("~").str[0].astype(int)
    modify_df["EndTime"] = modify_df["TimeGroup"].str.split("~").str[1].astype(int)
    modify_df = modify_df[(modify_df["StartTime"] >=start_interval) &(modify_df["EndTime"] <= end_use_interval)]
    modify_df = modify_df[~modify_df["New_Measurement"].isin([266, 901, 902, 903, 904, 905])]

    modify_df.sort_values(["StartTime", "New_Measurement"], inplace=True)

    return modify_df


def variable_timegroup_avg(stvm_df):
    copy_df = stvm_df.copy()
    variable_time_df = copy_df.groupby("TimeGroup")[["delta_V", "delta_K", "rate", "Phi_진입", "RFR", "F(outrate)"]].mean()
    return variable_time_df

def variable_total_avg(variable_df):
    variable_total_df = pd.DataFrame([variable_df.mean(numeric_only=True)])
    return variable_total_df

def speed_density_avg(density_df):
    copy_df = density_df.copy()
    avg_df = modify_frame(copy_df)
    avg_df = pd.DataFrame([avg_df.mean(numeric_only=True)])
    avg_df = avg_df[["V_mean", "K"]]
    return avg_df

def pivot_table(df, value, preprocess=None):
    copy_df = df.copy()
    if preprocess :
        copy_df = preprocess(copy_df)
    copy_df = copy_df.pivot(index="TimeGroup", columns="New_Measurement", values=value)

    copy_df = (
        copy_df
        .assign(_t=lambda x: x.index.astype(str).str.split("~").str[0].astype(int))
        .sort_values("_t")
        .drop(columns="_t")
    )
    copy_df = copy_df.fillna(0)
    return copy_df

def excel_color(val):
    if pd.isna(val):
        return ""
    elif val <= 0:
        return "background-color: #FF0000" # 빨간색
    else:
        return "background-color: #FFC000"  # 주황색


def weighted_avg_speed(original_df):
    copy_df = original_df.copy()
    # TimeGroup, New_Measurement별 그룹화 및 속도 평균
    speed_mean_df = (
        copy_df.groupby(["TimeGroup", "New_Measurement", "Vehicle type"])
          .agg(V_mean=("v[km/h]", "mean"), V_count=("v[km/h]", "count"))
          .reset_index()
    )
    speed_mean_df["std_group"] = speed_mean_df.groupby(["TimeGroup", "New_Measurement"])["V_mean"].transform(lambda s: s.std(ddof=0))
    speed_mean_df["cv"] = speed_mean_df["std_group"] / speed_mean_df["V_mean"]
    speed_mean_df["w"] = 1 / speed_mean_df["cv"]
    speed_mean_df["w*v"] = speed_mean_df["w"] * speed_mean_df["V_mean"]

    weighted_result = (
        speed_mean_df.groupby(["TimeGroup","New_Measurement"])
          .apply(lambda g: g["w*v"].sum() / g["w"].sum())
          .reset_index(name="Weighted_Avg_Speed")
    )

    return weighted_result

def save_to_excel(excel_df, folder_path, file_name, color=False):
        result_folder = os.path.join(folder_path, "해상도+모형식 수정")
        os.makedirs(result_folder, exist_ok=True)
        excel_file_name = f"{file_name}.xlsx"
        excel_file_path = os.path.join(result_folder, excel_file_name)

        if color:
            styled = excel_df.style.applymap(excel_color)
            styled.to_excel(excel_file_path, engine="openpyxl")
        else:
            excel_df.to_excel(excel_file_path, index=True)

        print(f"{excel_file_name} 생성 완료")

def z_to_score(z, z_min, z_max):
    if 1.645 <= z <= z_max:
        return 50 + ((95 + 5 * ((z - 1.645) / (z_max - 1.645))) * 0.5)
    elif 1.282 <= z < 1.645:
        return 50 + ((90 + 5 * ((z - 1.282) / (1.645 - 1.282))) * 0.5)
    elif 1.038 <= z < 1.282:
        return 50 + ((85 + 5 * ((z - 1.038) / (1.282 - 1.038))) * 0.5)
    elif 0.842 <= z < 1.038:
        return 50 + ((80 + 5 * ((z - 0.842) / (1.038 - 0.842))) * 0.5)
    elif 0.676 <= z < 0.842:
        return 50 + ((75 + 5 * ((z - 0.676) / (0.842 - 0.676))) * 0.5)
    elif 0.526 <= z < 0.676:
        return 50 + ((70 + 5 * ((z - 0.526) / (0.676 - 0.526))) * 0.5)
    elif 0.387 <= z < 0.526:
        return 50 + ((65 + 5 * ((z - 0.387) / (0.526 - 0.387))) * 0.5)
    elif 0.255 <= z < 0.387:
        return 50 + ((60 + 5 * ((z - 0.255) / (0.387 - 0.255))) * 0.5)
    elif -0.255 <= z < 0.255:
        return 50 + ((40 + 5 * ((z + 0.255) / (0.255 + 0.255))) * 0.5)
    elif -0.387 <= z < -0.255:
        return 50 + ((35 + 5 * ((z + 0.387) / (-0.255 + 0.387))) * 0.5)
    elif -0.526 <= z < -0.387:
        return 50 + ((30 + 5 * ((z + 0.526) / (-0.387 + 0.526))) * 0.5)
    elif -0.676 <= z < -0.526:
        return 50 + ((25 + 5 * ((z + 0.676) / (-0.676 + 0.842))) * 0.5)
    elif -0.842 <= z < -0.676:
        return 50 + ((20 + 5 * ((z + 0.842) / (-0.676 + 0.842))) * 0.5)
    elif -1.038 <= z < -0.842:
        return 50 + ((15 + 5 * ((z + 1.038) / (-0.842 + 1.038))) * 0.5)
    elif -1.282 <= z < -1.038:
        return 50 + ((10 + 5 * ((z + 1.282) / (-1.038 + 1.282))) * 0.5)
    elif -1.645 <= z < -1.282:
        return 50 + ((5 + 5 * ((z + 1.645) / (-1.282 + 1.645))) * 0.5)
    elif z_min <= z < -1.645:
        return 50 + ((0 + 5 * ((z + z_min) / (-1.645 + z_min))) * 0.5)
    else:
        return np.nan

def get_upstream_detector(root, location):
    detector_no = None
    detector_link = None
    gradient = None

    rsa_lane = None
    rsa_pos = None

    # 초기 데이터
    best_pos = -1

    for rsa in root.findall(".//reducedSpeedArea"):
        if rsa.get("name") == location:

            rsa_lane = rsa.get("lane")
            rsa_pos = float(rsa.get("pos"))

            # 유고 지점과 가장 가까운 상류 검지기
            for dcp in root.findall(".//dataCollectionPoint"):
                if dcp.get("lane") == rsa_lane:
                    dcp_pos = float(dcp.get("pos"))
                    if rsa_pos >= dcp_pos >= best_pos:
                        best_pos = dcp_pos

                        # 검지기 번호 가공 10010 -> 10
                        detector_no = int(dcp.get("no")) % 1000 - up_dcp_pos
                        detector_link = detector_map.get(detector_no)


    if detector_no is None:
        print("링크 번호가 다름")
        best_pos = float("inf")

        for dcp in root.findall(".//dataCollectionPoint"):
            if dcp.get("lane") == rsa_lane:
                dcp_pos = float(dcp.get("pos"))
                if rsa_pos < dcp_pos < best_pos:
                    best_pos = dcp_pos
                    detector_no = int(dcp.get("no")) % 1000 - 1- up_dcp_pos
                    detector_link = detector_map.get(detector_no)

    for link in root.findall(".//link"):
        if int(link.get("no")) == detector_link:
            gradient = round(float(link.get("gradient")),4)
            break

    return detector_no, gradient

def calculate_logD(speed_avg, entry_avg, heavy_avg, stvm_avg):
    speed_df = speed_avg.copy()
    entry_df = entry_avg.copy()
    heavy_df = heavy_avg.copy()
    stvm_df = stvm_avg.copy()

    speed_after = speed_df[(speed_df["New_Measurement"] == incident_measurement) & (speed_df["StartTime"] >= acc_start_time)].sort_values("StartTime").reset_index(drop=True)
    speed_after["is_congested"] = speed_after["V_mean"] <= Vc
    T2 = None


    for i in range(len(speed_after) - continuous_n + 1):

        if speed_after.loc[i:i + continuous_n - 1, "is_congested"].all():
            T2 = speed_after.loc[i + continuous_n -1 , "EndTime"]
            break

    if T2 is None:
        dc = np.nan
    else:
        dc = (T2 - acc_start_time) / 60
        if dc == 0:
            dc = np.nan
    print("dc : ", dc)

    el = lane - acc_lane

    before_group = f"{acc_start_time-sec_time}~{acc_start_time}"
    after_group = f"{acc_start_time}~{acc_start_time+sec_time}"

    s_before = entry_df.loc[(entry_df["TimeGroup"] == before_group) & (entry_df["New_Measurement"] == incident_measurement), "Phi_진입"].iloc[0]

    hv = heavy_df.loc[(heavy_df["TimeGroup"] == after_group) & (heavy_df["New_Measurement"] == incident_measurement), "rate"].iloc[0]

    stvm_before = stvm_df.loc[(stvm_df["TimeGroup"] == before_group) & (stvm_df["New_Measurement"] == incident_measurement),"STVM"].iloc[0]
    stvm_after = stvm_df.loc[(stvm_df["TimeGroup"] == after_group) & (stvm_df["New_Measurement"] == incident_measurement), "STVM"].iloc[0]

    log_dc_df = pd.DataFrame({
        # 종속 변수
        "Dc" : dc,
        "log_Dc" : np.log(dc),

        # 구조, 운영 변수
        "EL": el,
        "T": (acc_finish_time - acc_start_time)/60, # 분 단위
        "S_before": s_before,
        "HV": hv,
        "G": lane_gradient,

        # 교통류 반응 변수
        "STVM": stvm_before,
        "Delta_STVM": stvm_after - stvm_before,

    }, index=[0])
    return log_dc_df

def centralization_log_d(df):
    df = df.copy()
    means = df[["EL", "STVM", "Delta_STVM", "S_before", "HV", "G"]].mean(numeric_only=True)

    df["EL_c"] = df["EL"] - means["EL"]
    df["I_EL"] = (df["EL"] > 0).astype(int)
    df["T_c"] = df["T"]
    df["S_c"] = df["S_before"] - means["S_before"]
    df["HV_c"] = df["HV"] - means["HV"]
    df["G_c"] = df["G"] - means["G"]
    df["S_c**2"] = df["S_c"] ** 2
    df["STVM_c"] = df["STVM"] - means["STVM"]
    df["Delta_STVM_c"] = df["Delta_STVM"] - means["Delta_STVM"]
    df["EL_c*S_c"] = df["EL_c"] * df["S_c"]
    df["HV_c*G_c"] = df["HV_c"] * df["G_c"]

    df["I_EL_S"] = df["I_EL"] * df["S_c"]

    result_df = df[["Dc", "log_Dc","EL_c", "I_EL", "T_c", "S_c", "HV_c", "G_c", "S_c**2", "STVM_c", "Delta_STVM_c", "EL_c*S_c", "HV_c*G_c", "I_EL_S"]]

    result_df = result_df.round(3)
    return result_df


def complete_time_measurement(df, value_cols, measurements=None):
    df = df.copy()

    # 전체 시간대 생성
    full_time = pd.DataFrame({
        "StartTime": np.arange(start_interval, end_interval, sec_time)
    })
    full_time["EndTime"] = full_time["StartTime"] + sec_time
    full_time["TimeGroup"] = (
        full_time["StartTime"].astype(str)
        + "~"
        + full_time["EndTime"].astype(str)
    )

    # 전체 검지기 목록
    if measurements is None:
        measurements = sorted(df["New_Measurement"].dropna().unique())

    measurement_df = pd.DataFrame({
        "New_Measurement": measurements
    })

    full_index = (
        full_time.assign(key=1)
        .merge(measurement_df.assign(key=1), on="key")
        .drop(columns="key")
    )

    # 누락된 TimeGroup × New_Measurement 생성
    df = full_index.merge(
        df,
        on=["TimeGroup", "New_Measurement"],
        how="left"
    )

    # 값 없는 부분 0 처리
    for col in value_cols:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    df = sort_ascending(df)

    return df
###################################################################################################################

senario_df = pd.read_csv(senario_path, encoding="cp949")

tree = ET.parse(inpx_path)
root = tree.getroot()


# 검지기 정보 미리 로드
detector_map = {}

for dcp in root.findall(".//dataCollectionPoint"):

    # 검지기 번호 (10010 → 10)
    detector_no = int(dcp.get("no")) % 1000

    # 링크 번호 ("23 1" → 23)
    detector_link = int(dcp.get("lane").split()[0])

    # 딕셔너리에 저장
    detector_map[detector_no] = detector_link


folder_path = path
parquet_list = sorted([file for file in os.listdir(folder_path) if file.endswith(".parquet")])

for sec_time in sec_time_list:

    revision_time = 3600 / sec_time

    for up_dcp_pos in up_dcp_pos_list:

        gradient_list = []

        print(f"\n===== 집계시간 {sec_time}초 / 상류검지기 {up_dcp_pos} =====")

        for cdx, continuous_num in enumerate(continuous_list):
            continuous_n = int(continuous_num / sec_time)
            log_d_df_list = []
            vc_result_list = []
            for idx, start in enumerate(range(0, len(parquet_list), num_mer)):
                print(f"============ idx={idx}, start={start} ============")

                row = senario_df.iloc[idx]

                # 기존 차로 수
                lane = row["lane_count"]

                # 유고 차로 수
                acc_lane = row["lane_closure_count"]

                # 유고 종료 시간
                acc_finish_time = acc_start_time + row["incident_duration_min"] * 60

                # 유고 지점
                incident_location = row["location_name"]
                locations = [x.strip() for x in incident_location.split(",")] # [본선1, 본선2]
                location = locations[0]
                incident_measurement, lane_gradient = get_upstream_detector(root, location)

                gradient_list.append((incident_measurement, lane_gradient))

                batch_files = parquet_list[start:start + num_mer]

                print("Vc : ", Vc)
                print(
                     f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] "
                    f"입력값 | 교통량={row['base_main_in_vph']}, 기존 차로수={lane}, 유고 차로수={acc_lane}, 유고지속시간={row['incident_duration_min']}, 유고지점={incident_location}, 검지기={incident_measurement}, lane_gradient={lane_gradient}"
                    )

                i = start // num_mer

                speed_list = []
                density_list = []
                heavy_list = []
                entry_list = []
                rfr_list_all = []
                normality_list_all = []
                stvm_list = []
                vc_speed_list = []

                for index, mer_file in enumerate(batch_files):
                    print("작업파일 :", mer_file)

                    file_path = os.path.join(folder_path, mer_file)
                    df = pd.read_parquet(file_path)

                    # 컬럼 내부 데이터 정수형 변환
                    df = df.apply(pd.to_numeric, errors="coerce")

                    original_df = df[(df["t(Entry)"] != -1.00)].reset_index(drop=True)

                    # 불필요 컬럼 제거
                    original_df.drop(columns=["b[m/s2]", "tQueue", "Occ", "Pers"], inplace=True, errors="ignore")

                    # 차로 통합을 위한 컬럼
                    original_df["New_Measurement"] = original_df["Measurem."] % 1000

                    # 원본 데이터 vc 도출
                    vc_speed_list.append(original_df[["New_Measurement", "t(Entry)", "v[km/h]"]])

                    # 5분단위 집계
                    bins = np.arange(start_interval, end_interval+1, sec_time)
                    labels = [f"{start}~{start+sec_time}" for start in bins[:-1]]  # 구간 라벨링

                    # 구간 나누기 및 컬럼 추가
                    original_df["TimeGroup"] = pd.cut(original_df["t(Entry)"], bins=bins, labels=labels, right=False)

                    original_df = original_df[original_df["TimeGroup"].notna()].copy()
                    original_df["TimeGroup"] = original_df["TimeGroup"].astype(str)

                    # 가감속 차로 부분 검지기 제거
                    #original_df = original_df[~original_df["Measurem."].between(30000, 39999)]

                    # 속도변화율
                    speed_df = speed_mean(original_df)
                    speed_list.append(speed_df)


                    # 밀도변화율
                    density_df = density_mean(speed_df)
                    density_list.append(density_df)

                    # 중차량 혼입률
                    heavy_df = heavy_rate(original_df)
                    heavy_list.append(heavy_df)

                    # 동적 포화도
                    entry_saturation_df = entry_saturation(original_df)
                    entry_list.append(entry_saturation_df)

                    # 램프 간섭 영향률
                    rfr_df = rfr_rate(original_df)
                    rfr_list_all.append(rfr_df)

                    # 진출 원활율
                    normality_df = output_normality(original_df)
                    normality_list_all.append(normality_df)

                    # STVM 계산
                    stvm_df = calculate_stvm(speed_df, density_df, heavy_df, entry_saturation_df, rfr_df, normality_df)
                    stvm_list.append(stvm_df)

                print("시드 평균 계산 중...")

                merged_speed = pd.concat(speed_list)
                merged_density = pd.concat(density_list)
                merged_heavy = pd.concat(heavy_list)
                merged_entry = pd.concat(entry_list)
                merged_rfr = pd.concat(rfr_list_all)
                merged_normality = pd.concat(normality_list_all)
                merged_stvm = pd.concat(stvm_list)
                vc_speed_df = pd.concat(vc_speed_list)

                # 각 지표 평균
                speed_avg = (
                    merged_speed
                    .groupby(["StartTime", "EndTime", "TimeGroup", "New_Measurement"])
                    [["V_mean", "delta_V"]]
                    .mean()
                    .reset_index()
                )

                density_avg = (
                    merged_density
                    .groupby(["StartTime","TimeGroup", "New_Measurement"])
                    [["delta_K"]]
                    .mean()
                    .reset_index()
                )

                heavy_avg = (
                    merged_heavy
                    .groupby(["StartTime", "TimeGroup", "New_Measurement"])
                    [["rate"]]
                    .mean()
                    .reset_index()
                )

                entry_avg = (
                    merged_entry
                    .groupby(["StartTime", "TimeGroup", "New_Measurement"])
                    [["Phi_진입"]]
                    .mean()
                    .reset_index()
                )

                rfr_avg = (
                    merged_rfr
                    .groupby(["StartTime", "TimeGroup", "New_Measurement"])
                    [["RFR"]]
                    .mean()
                    .reset_index()
                )

                normality_avg = (
                    merged_normality
                    .groupby(["StartTime", "TimeGroup", "New_Measurement"])
                    [["F(outrate)"]]
                    .mean()
                    .reset_index()
                )

                stvm_avg = (
                    merged_stvm
                    .groupby(["TimeGroup", "New_Measurement"], sort=False)
                    .mean(numeric_only=True)
                    .reset_index()
                )


                #save_to_excel(speed_avg, folder_path, "속도변화량", i)
                #save_to_excel(density_avg, folder_path, "밀도변화량", i)
                #save_to_excel(stvm_avg, folder_path, "STVM", i)

                # Z-Score 계산
                #z_score_df = calculate_z_score(stvm_avg)
                #save_to_excel(z_score_df, folder_path, f"환산점수", i)

                # STVM 피봇
                #stvm_pivot_df = pivot_table(stvm_avg, "STVM", preprocess=modify_frame)
                #save_to_excel(stvm_pivot_df, folder_path, f"지표합산값", i, color=True)

                # 속도값 피봇
                #speed_pivot_df = pivot_table(speed_avg, "V_mean", preprocess=modify_frame)
                #save_to_excel(speed_pivot_df, folder_path, "속도 피봇", i, color=True)

                # 임계 붕괴 시간 추정 모형
                log_d_df = calculate_logD(speed_avg, entry_avg, heavy_avg, stvm_avg)
                #display("log_d_df : ", log_d_df)
                log_d_df_list.append(log_d_df)
                #save_to_excel(log_d_df, folder_path, "임계추정모형", i)

                # 지표별 구성값(속도변화값)
                #speed_pivot_df = pivot_table(speed_df, "delta_V", preprocess=modify_frame)
                #save_to_excel(speed_pivot_df, folder_path, "속도변화값", i)

                # 지표별 구성값(밀도변화값)
                #density_pivot_df = pivot_table(density_df, "delta_K", preprocess=modify_frame)
                #save_to_excel(density_pivot_df, folder_path, "밀도변화값", i)

                # 지표별 구성값(중차량혼입률)
                #heavy_pivot_df = pivot_table(heavy_df, "rate", preprocess=modify_frame)
                #save_to_excel(heavy_pivot_df, folder_path, "중차량혼입률", i)

                # 지표별 구성값(동적포화도)
                #entry_saturation_pivot_df = pivot_table(entry_saturation_df, "Phi_진입", preprocess=modify_frame)
                #save_to_excel(entry_saturation_pivot_df, folder_path, "동적포화도", i)

                # 지표별 구성값(램프 간섭 영향률)
                #rfr_pivot_df = pivot_table(rfr_df, "RFR", preprocess=modify_frame)
                #save_to_excel(rfr_pivot_df, folder_path, "램프간섭영향률", i)

                # 지표별 구성값(진출 원활률)
                #normality_pivot_df = pivot_table(normality_df, "F(outrate)", preprocess=modify_frame)
                #save_to_excel(normality_pivot_df, folder_path, "진출원활률", i)

                # 메모리 정리
                #del df, original_df, speed_df, density_df, heavy_df, entry_saturation_df, rfr_df, normality_df, stvm_df, z_score_df
                gc.collect()

            log_d_df_all = pd.concat(log_d_df_list, ignore_index=True)
            final_log_d = centralization_log_d(log_d_df_all)
            final_log_d = final_log_d.fillna("NA")
            save_to_excel(final_log_d, folder_path, f"임계연속시간_영향권3_진출_{sec_time}초_{150 + (up_dcp_pos - 1) * 100}m")
            #gradient_df = pd.DataFrame(gradient_list, columns=["검지기 번호", "종단경사"])
            #save_to_excel(gradient_df, folder_path, "종단경사", 0)
            #save_to_excel(vc_result_df, folder_path, "임계속도", 0)


===== 집계시간 10초 / 상류검지기 1 =====
============ idx=0, start=0 ============
Vc :  53.7
[2026-06-18 00:06:20] 입력값 | 교통량=1444, 기존 차로수=3, 유고 차로수=1, 유고지속시간=10, 유고지점=영향권3(진출)1-1, 검지기=124, lane_gradient=-0.007
작업파일 : 화성~서울(110-영향권3_진출_3차로폐쇄-추정용)_260615_001.parquet
작업파일 : 화성~서울(110-영향권3_진출_3차로폐쇄-추정용)_260615_002.parquet
작업파일 : 화성~서울(110-영향권3_진출_3차로폐쇄-추정용)_260615_003.parquet
시드 평균 계산 중...
dc :  nan
============ idx=1, start=3 ============
Vc :  53.7
[2026-06-18 00:07:34] 입력값 | 교통량=1619, 기존 차로수=3, 유고 차로수=1, 유고지속시간=30, 유고지점=영향권3(진출)1-1, 검지기=124, lane_gradient=-0.007
작업파일 : 화성~서울(110-영향권3_진출_3차로폐쇄-추정용)_260615_004.parquet
작업파일 : 화성~서울(110-영향권3_진출_3차로폐쇄-추정용)_260615_005.parquet
작업파일 : 화성~서울(110-영향권3_진출_3차로폐쇄-추정용)_260615_006.parquet
시드 평균 계산 중...
dc :  nan
============ idx=2, start=6 ============
Vc :  53.7
[2026-06-18 00:08:58] 입력값 | 교통량=1118, 기존 차로수=3, 유고 차로수=3, 유고지속시간=60, 유고지점=영향권3(진출)1-1,영향권3(진출)1-2,영향권3(진출)1-3, 검지기=124, lane_gradient=-0.007
작업파일 : 화성~서울(110-영향권3_진출_3차로폐쇄-추정용)_260615_007.parquet
작업파일